# Spaceship Titanic: Passenger Transport Prediction

**Goal:** Predict the `Transported` target for passengers in the Kaggle Spaceship Titanic test set.

This notebook loads the original competition CSV files, explores missing values, engineers features, compares four classifiers on a stratified holdout validation set, and creates `submission.csv`.

**Reproducibility:** Random seed = 42. The validation score is an internal holdout result, not a Kaggle leaderboard score.

In [6]:
!unzip -q /content/Spaceship_Titanic_Project_Package.zip -d /content
import os
os.chdir("/content/spaceship_titanic_project")

print("Project data loaded:")
print(os.listdir("data"))

replace /content/spaceship_titanic_project/model_comparison.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
Project data loaded:
['sample_submission.csv', 'test.csv', 'train.csv']


In [13]:
import pandas as pd

submission = pd.read_csv("submission.csv")

print(submission.head())
print("Rows:", len(submission))
print("Columns:", submission.columns.tolist())
print("Missing values:", submission.isnull().sum().sum())
print("Predictions:", submission["Transported"].value_counts())

  PassengerId  Transported
0     0013_01         True
1     0018_01        False
2     0019_01         True
3     0021_01         True
4     0023_01         True
Rows: 4277
Columns: ['PassengerId', 'Transported']
Missing values: 0
Predictions: Transported
False    2228
True     2049
Name: count, dtype: int64


In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.base import clone

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
sample_submission = pd.read_csv("data/sample_submission.csv")

print("Train:", train.shape)
print("Test:", test.shape)
print(train.head())

Train: (8693, 14)
Test: (4277, 13)
  PassengerId HomePlanet CryoSleep  Cabin  Destination   Age    VIP  \
0     0001_01     Europa     False  B/0/P  TRAPPIST-1e  39.0  False   
1     0002_01      Earth     False  F/0/S  TRAPPIST-1e  24.0  False   
2     0003_01     Europa     False  A/0/S  TRAPPIST-1e  58.0   True   
3     0003_02     Europa     False  A/0/S  TRAPPIST-1e  33.0  False   
4     0004_01      Earth     False  F/1/S  TRAPPIST-1e  16.0  False   

   RoomService  FoodCourt  ShoppingMall     Spa  VRDeck               Name  \
0          0.0        0.0           0.0     0.0     0.0    Maham Ofracculy   
1        109.0        9.0          25.0   549.0    44.0       Juanna Vines   
2         43.0     3576.0           0.0  6715.0    49.0      Altark Susent   
3          0.0     1283.0         371.0  3329.0   193.0       Solam Susent   
4        303.0       70.0         151.0   565.0     2.0  Willy Santantines   

   Transported  
0        False  
1         True  
2        False  
3

## 1. Initial data inspection

The training set has 8,693 rows and includes the target `Transported`. The test set has 4,277 rows and omits that target. `PassengerId` is retained for the final output.

In [8]:
display(train.info())
display(train.isna().sum().sort_values(ascending=False))
display(train["Transported"].value_counts(normalize=True))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   object 
 1   HomePlanet    8492 non-null   object 
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   object 
 4   Destination   8511 non-null   object 
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   object 
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(7)
memory usage: 891.5+ KB


None

,0
CryoSleep,217
ShoppingMall,208
VIP,203
HomePlanet,201
Name,200
Cabin,199
VRDeck,188
Spa,183
FoodCourt,183
Destination,182


,proportion
Transported,
True,0.503624
False,0.496376


## 2. Feature engineering

The model derives passenger group size from `PassengerId`, splits cabin into deck/number/side, sums onboard spending, flags passengers with no spending, and derives surname group size. Raw identifiers and names are removed before modelling.

In [9]:
def make_features(df):
    x = df.copy()
    x["GroupId"] = x["PassengerId"].str.split("_").str[0]
    x["GroupSize"] = x.groupby("GroupId")["PassengerId"].transform("count")
    cabin_parts = x["Cabin"].fillna("Unknown/0/Unknown").str.split("/", expand=True)
    x["CabinDeck"] = cabin_parts[0]
    x["CabinNumber"] = pd.to_numeric(cabin_parts[1], errors="coerce")
    x["CabinSide"] = cabin_parts[2]
    spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
    x["TotalSpend"] = x[spend_cols].sum(axis=1)
    x["NoSpend"] = (x["TotalSpend"] == 0).astype(int)
    x["Surname"] = x["Name"].fillna("Unknown Unknown").str.split().str[-1]
    x["SurnameGroupSize"] = x.groupby("Surname")["PassengerId"].transform("count")
    return x.drop(columns=["PassengerId", "Cabin", "Name", "GroupId", "Surname"], errors="ignore")

X = make_features(train.drop(columns=["Transported"]))
y = train["Transported"].astype(bool)
X_test = make_features(test)
print(X.shape, X_test.shape)

(8693, 17) (4277, 17)


## 3. Preprocessing and validation split

Numeric values are median-imputed and scaled. Categorical values are imputed with the most frequent category and one-hot encoded. A stratified 80/20 split is used for model comparison.

In [10]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
cat_cols = X.select_dtypes(include=["object", "bool"]).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols)
])

## 4. Compare models

Accuracy and F1-score are calculated on the same validation split. The model with the highest validation accuracy is selected for the final fit. This is a baseline comparison; further tuning may improve results.

In [11]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1500, C=0.7),
    "Random Forest": RandomForestClassifier(
        n_estimators=350, min_samples_leaf=2, max_features="sqrt",
        random_state=42, n_jobs=-1
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=400, min_samples_leaf=2, max_features="sqrt",
        random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=180, learning_rate=0.05, max_depth=2, random_state=42
    )
}

results = []
fitted_models = {}
for name, estimator in models.items():
    pipeline = Pipeline([("prep", clone(preprocessor)), ("model", estimator)])
    pipeline.fit(X_train, y_train)
    validation_predictions = pipeline.predict(X_val)
    results.append({
        "Model": name,
        "Validation Accuracy": accuracy_score(y_val, validation_predictions),
        "Validation F1": f1_score(y_val, validation_predictions)
    })
    fitted_models[name] = pipeline

comparison = pd.DataFrame(results).sort_values(
    "Validation Accuracy", ascending=False
).reset_index(drop=True)
display(comparison)
comparison.to_csv("model_comparison.csv", index=False)

,Model,Validation Accuracy,Validation F1
0,Random Forest,0.810811,0.808832
1,Extra Trees,0.799310,0.790138
2,Gradient Boosting,0.797010,0.799318
3,Logistic Regression,0.790684,0.794350


## 5. Train selected model and create Kaggle submission

The selected model is retrained on all labelled training rows, then predicts the test set. The output must contain exactly `PassengerId` and `Transported`, with 4,277 prediction rows.

In [12]:
best_name = comparison.loc[0, "Model"]
final_model = Pipeline([
    ("prep", clone(preprocessor)),
    ("model", models[best_name])
])
final_model.fit(X, y)
test_predictions = final_model.predict(X_test).astype(bool)

submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Transported": test_predictions
})
assert submission.shape == (4277, 2)
assert submission["PassengerId"].equals(sample_submission["PassengerId"])
submission.to_csv("submission.csv", index=False)

print("Selected model:", best_name)
print("Submission shape:", submission.shape)
display(submission.head())
print("Saved submission.csv")

Selected model: Random Forest
Submission shape: (4277, 2)


,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True


Saved submission.csv


## 6. Interpretation and limitations

The best holdout result in this run was Random Forest with **81.08% validation accuracy** and **0.8088 F1-score**. This is not the Kaggle public leaderboard score. Results depend on the chosen random split and feature engineering. The current approach uses a single holdout split rather than cross-validation, and additional tuning or leakage-aware family/group features may change performance.

The notebook uses the provided competition data. The Kaggle dataset and competition rules should be acknowledged in the report. AI assistance used during development should be declared according to the university template.